In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import skew
from sklearn.preprocessing import RobustScaler

# 1. 讀取資料
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

# ==========================================
# 2. 移除離群值 (強化版：嚴格濾除市場雜訊)
# ==========================================
# 條件 1: 面積大於 4000 的巨型房屋 (原作者強烈建議全部刪除，因為不具備市場代表性)
# 條件 2: 品質極差 (Qual<5) 卻賣天價 (>20萬) 的雜訊
# 條件 3: 品質極佳 (Qual>9) 卻賣破盤價 (<15萬) 的親友交易雜訊

outliers = train[
    (train["GrLivArea"] > 4000) | 
    ((train["OverallQual"] < 5) & (train["SalePrice"] > 200000)) | 
    ((train["OverallQual"] > 9) & (train["SalePrice"] < 150000))
].index

train = train.drop(outliers)
print(f"已移除 {len(outliers)} 筆離群雜訊資料！")

# 3. 目標變數轉換 (對數轉換以處理偏態)
y_train = np.log1p(train["SalePrice"])

# 將 train 與 test 合併進行特徵工程 (捨棄不必要的 Id 欄位)
train_features = train.drop(["SalePrice", "Id"], axis=1)
test_features = test.drop(["Id"], axis=1)
all_data = pd.concat([train_features, test_features]).reset_index(drop=True)

# ==========================================
# 4. 數值型態修正 (將「數字代表類別」的變數轉為字串)
# ==========================================
for col in ['MSSubClass', 'MoSold', 'YrSold']:
    all_data[col] = all_data[col].apply(str)

# 5. 缺失值填補
# 依據同一個 Neighborhood 的 LotFrontage 中位數填補
all_data["LotFrontage"] = all_data.groupby("Neighborhood")["LotFrontage"].transform(
    lambda x: x.fillna(x.median())
)

numeric_cols = all_data.select_dtypes(exclude=['object']).columns
categorical_cols = all_data.select_dtypes(include=['object']).columns

# 數值特徵填補 0，類別特徵填補 None
all_data[numeric_cols] = all_data[numeric_cols].fillna(0)
all_data[categorical_cols] = all_data[categorical_cols].fillna("None")
# 修正車庫年份缺失值：若無車庫，則用房屋建造年份填補
all_data["GarageYrBlt"] = all_data["GarageYrBlt"].fillna(all_data["YearBuilt"])

# ==========================================
# 6. 進階特徵工程
# ==========================================
# ==========================================
# 6. 進階特徵工程 
# ==========================================
all_data["TotalSF"] = (
    all_data["TotalBsmtSF"] + all_data["1stFlrSF"] + all_data["2ndFlrSF"]
)
all_data["HouseAge"] = all_data["YrSold"].astype(int) - all_data["YearBuilt"]
all_data["RemodelAge"] = (
    all_data["YrSold"].astype(int) - all_data["YearRemodAdd"]
)

# 假設翻修年份對買家的影響力佔 60%，原始建造年份佔 40%
all_data["EffectiveAge"] = (all_data["HouseAge"] * 0.4) + (all_data["RemodelAge"] * 0.6)

# 釋放線性模型的共線性壓力
all_data = all_data.drop(columns=["YearBuilt", "YearRemodAdd"])

all_data["TotalBath"] = (
    all_data["FullBath"]
    + (0.5 * all_data["HalfBath"])
    + all_data["BsmtFullBath"]
    + (0.5 * all_data["BsmtHalfBath"])
)


# 整體品質 x 總建築面積
all_data["Qual_TotalSF"] = all_data["OverallQual"] * all_data["TotalSF"]
# 總門廊面積（把散落的各種Porch加總，房子外觀爽度指標）
all_data["TotalPorchSF"] = (
    all_data["OpenPorchSF"]
    + all_data["EnclosedPorch"]
    + all_data["3SsnPorch"]
    + all_data["ScreenPorch"]
)

all_data["TotalOutdoorSpace"] = (
    all_data["WoodDeckSF"]
    + all_data["OpenPorchSF"]
    + all_data["EnclosedPorch"]
    + all_data["3SsnPorch"]
    + all_data["ScreenPorch"]
)

# 衛浴總數與房間數的比例（衡量房屋設計是否合理，如：5間房只有1個衛浴會扣分）
abv_baths = all_data["FullBath"] + (0.5 * all_data["HalfBath"])
all_data["BathToRoomRatio"] = abv_baths / (all_data["TotRmsAbvGrd"] + 1e-5)# 加上1e-5防止分母為0

all_data["OverallScore"] = (
    all_data["OverallQual"]
    * all_data["OverallCond"]
)

all_data["GarageScore"] = (
    all_data["GarageCars"]
    * all_data["GarageArea"]
)

all_data["TotalHomeQuality"] = (
    all_data["OverallQual"]
    + all_data["OverallCond"]
)

all_data["QualGrLivArea"] = (
    all_data["OverallQual"]
    * all_data["GrLivArea"]
)

all_data["TotalSF2"] = (
    all_data["TotalSF"] ** 2
)


# 二元特徵標籤
all_data['HasPool'] = all_data['PoolArea'].apply(lambda x: 1 if x > 0 else 0)
all_data['Has2ndFloor'] = all_data['2ndFlrSF'].apply(lambda x: 1 if x > 0 else 0)
all_data['HasGarage'] = all_data['GarageArea'].apply(lambda x: 1 if x > 0 else 0)
all_data['HasBsmt'] = all_data['TotalBsmtSF'].apply(lambda x: 1 if x > 0 else 0)
all_data['HasFireplace'] = all_data['Fireplaces'].apply(lambda x: 1 if x > 0 else 0)

# 將月份轉回數值以進行三角函數轉換
all_data["MoSold_Num"] = all_data["MoSold"].astype(int)

# 週期性特徵轉換
all_data["MoSold_Sin"] = np.sin(2 * np.pi * all_data["MoSold_Num"] / 12)
all_data["MoSold_Cos"] = np.cos(2 * np.pi * all_data["MoSold_Num"] / 12)

# 轉換完畢後，可以將原始的 MoSold_Num 刪除
all_data = all_data.drop(["MoSold_Num", "MoSold"], axis=1)

# 【零洩漏安全版】：改用 OverallQual 作為社區分級的代理指標
neighborhood_median = train.groupby("Neighborhood")["OverallQual"].median()

# 根據中位數由低到高排序，並給予 1 到 N 的等級分數
neighborhood_rank = neighborhood_median.sort_values().index
neighborhood_mapping = {n: i for i, n in enumerate(neighborhood_rank, 1)}

# 映射回 all_data 中建立新特徵
all_data["Neighborhood_Rank"] = all_data["Neighborhood"].map(neighborhood_mapping)

# 請在第 6 步的最尾端，把這行 drop 補進去：
all_data = all_data.drop(columns=["TotalPorchSF"])

# === 新增特徵區塊 1：空間與地段進階指標 ===
# 容積率：數值越低代表庭院比例越大，居住舒適度可能越高
all_data['LivingArea_to_Lot_Ratio'] = all_data['GrLivArea'] / (all_data['LotArea'] + 1e-5)

# 實際戶外庭院面積：總地坪扣掉一樓與車庫佔地
all_data['Actual_Yard_Area'] = all_data['LotArea'] - all_data['1stFlrSF'] - all_data['GarageArea']

# 未裝潢地下室佔比：衡量地下室是否具備休閒價值
all_data['UnfBsmt_Ratio'] = all_data['BsmtUnfSF'] / (all_data['TotalBsmtSF'] + 1e-5)

# 嫌惡設施與地段溢價標籤：將分散的 Condition 綜合評估
premium_conditions = ['PosN', 'PosA'] 
penalty_conditions = ['Artery', 'Feedr', 'RRNn', 'RRAn', 'RRMn', 'RRNe'] 
all_data['Has_Premium_Location'] = all_data[['Condition1', 'Condition2']].isin(premium_conditions).any(axis=1).astype(int)
all_data['Has_Penalty_Location'] = all_data[['Condition1', 'Condition2']].isin(penalty_conditions).any(axis=1).astype(int)

# 刪除「幾乎沒有變異度」的廢特徵，減輕 Lasso 的共線性與雜訊壓力
# Utilities: 幾乎 100% 都是 'AllPub'
# Street: 99.6% 都是 'Pave' (柏油路)
# PoolQC: 99.6% 都是缺失值 (沒游泳池)
noise_cols = ["Utilities", "Street", "PoolQC"]
all_data = all_data.drop(columns=noise_cols, errors="ignore")

# ==========================================
# 7. 順序型特徵轉換 (Ordinal Encoding) - 讓 XGBoost 變精準的關鍵！
# ==========================================
qual_mapping = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
qual_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
             'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond']

for col in qual_cols:
    all_data[col] = all_data[col].map(qual_mapping)
    
exposure_map = {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4}
all_data['BsmtExposure'] = all_data['BsmtExposure'].map(exposure_map)

# === 新增特徵區塊 2：品質與面積深度互動 ===
# 確保這些運算在 Ordinal Encoding 完成後執行，才能取得數值化的品質分數
all_data['Bsmt_Real_Value'] = all_data['TotalBsmtSF'] * all_data['BsmtQual']
all_data['Fireplace_Real_Value'] = all_data['Fireplaces'] * all_data['FireplaceQu']
all_data['Garage_Real_Value'] = all_data['GarageArea'] * all_data['GarageQual']

# 綜合豪華度分數：綜合考量外部品質、廚房品質與附加設施
all_data['Luxury_Score'] = all_data['ExterQual'] + all_data['KitchenQual'] + all_data['HasPool'] + all_data['HasFireplace']


# ==========================================
# 8. 處理數值特徵的偏態 (Skewness) 
# ==========================================
# 重新抓取更新後的數值欄位
numeric_cols = all_data.select_dtypes(exclude=['object']).columns

# 【助教的防呆機制】：過濾掉包含負數的特徵 (例如 Sin/Cos)，避免 np.log1p 產生 -inf
safe_numeric_cols = [col for col in numeric_cols if all_data[col].min() >= 0]

# 計算偏態 (只針對安全的數值特徵)
skewed_feats = all_data[safe_numeric_cols].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)

# 門檻篩選條件
high_skew = skewed_feats[skewed_feats > 0.75].index

from scipy.special import boxcox1p
from scipy.stats import boxcox_normmax

for feat in high_skew:
    try:
        # 改進點 A：引入 method='mle'（極大似然估計法），它在面對大量重複值時，數學底層比預設的 pearsonr 穩定非常多
        opt_lambda = boxcox_normmax(all_data[feat] + 1, method="mle")
        all_data[feat] = boxcox1p(all_data[feat], opt_lambda)

    except Exception:
        # 改進點 B：工程師的終極防撞網。
        # 只要 Scipy 算不出 optimal lambda，我們就不跟它糾纏，直接悄悄降級回最安全的 np.log1p (等同於 λ=0 的 Box-Cox)
        all_data[feat] = np.log1p(all_data[feat])

print(f"已對 {len(high_skew)} 個偏態特徵進行優化轉換。")
# ==========================================
# 9. 類別特徵 One-Hot Encoding
# ==========================================
all_data = pd.get_dummies(all_data)
print(f"特徵工程與 Encoding 後的資料維度: {all_data.shape}")

# 10. 將資料切分回 train 與 test
X_train = all_data.iloc[:len(y_train), :]
X_test = all_data.iloc[len(y_train):, :]

# ==========================================
# 11. 特徵縮放 (RobustScaler 抵抗潛在極端值)
# ==========================================
# 保留原始資料給 Tree Model
X_train_tree = X_train.copy()
X_test_tree = X_test.copy()

# 給 Linear Model 使用（把非線性的分數項剔除）
linear_drop_cols = ["GarageScore", "OverallScore", "TotalSF2"]
X_train_lin = X_train.drop(columns=linear_drop_cols, errors="ignore")
X_test_lin = X_test.drop(columns=linear_drop_cols, errors="ignore")

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_lin)
X_test_scaled = scaler.transform(X_test_lin)

X_train = pd.DataFrame(X_train_scaled, columns=X_train_lin.columns)
X_test = pd.DataFrame(X_test_scaled, columns=X_test_lin.columns)

print(f"處理完畢！X_train 維度: {X_train.shape}, X_test 維度: {X_test.shape}")

已移除 5 筆離群雜訊資料！


C:\Users\austi\AppData\Local\Temp\ipykernel_19076\3188534181.py:47: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = all_data.select_dtypes(include=['object']).columns


已對 30 個偏態特徵進行優化轉換。
特徵工程與 Encoding 後的資料維度: (2914, 295)
處理完畢！X_train 維度: (1455, 292), X_test 維度: (1459, 292)


In [ ]:
X_train.to_csv(
    "../data/X_train.csv",
    index=False
)

X_test.to_csv(
    "../data/X_test.csv",
    index=False
)

pd.DataFrame(y_train).to_csv(
    "../data/y_train.csv",
    index=False
)

print("資料已儲存")

資料已儲存


In [ ]:
X_train_tree.to_csv(
    "../data/X_train_tree.csv",
    index=False
)

X_test_tree.to_csv(
    "../data/X_test_tree.csv",
    index=False
)